# 🤖 Machine Learning — Complete Notebook
## Linear Regression → Logistic Regression → Decision Tree

---

### Is Notebook Mein Kya Seekhoge?

| # | Topic | Kya Karenge |
|---|-------|-------------|
| 1 | **Linear Regression** | Numbers predict karna (marks, salary, price) |
| 2 | **Logistic Regression** | Yes/No predict karna (pass/fail, spam/not spam) |
| 3 | **Decision Tree** | Rules se decision lena (koi bhi prediction) |

### Har Topic Ka Structure:
```
Theory (kya hai, kaise kaam karta hai)
    ↓
Dataset banana
    ↓
ML Pipeline (Train/Test Split → Model → Predict → Evaluate)
    ↓
Visualization
    ↓
Key Takeaways
```

> **Note:** Ye notebook Statistics Lessons ke baad ki hai — Mean, Median, SD, Correlation, EDA — woh concepts yahan use honge!

---
## ⚙️ Setup — Libraries Install & Import

In [ ]:
# Ye sab Colab mein already installed hain
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Scikit-learn — ML ki main library
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (
    mean_absolute_error, mean_squared_error, r2_score,
    accuracy_score, confusion_matrix, classification_report
)
from sklearn.preprocessing import LabelEncoder

# Plot style
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11
sns.set_theme(style='whitegrid', palette='Set2')

np.random.seed(42)

print('✅ Sab libraries load ho gayi!')

---
# 🧠 ML Shuru Karne Se Pehle — 3 Zaroori Concepts

## Concept 1 — Features (X) aur Target (y)

```
Features (X)                    Target (y)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
study_hours | attendance | tv_hours  →  marks
     5      |     90     |    1      →   78
     2      |     70     |    4      →   52
     8      |     95     |    0      →   91
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
   INPUT  (jo hum jaante hain)    OUTPUT (jo predict karna hai)
```

- **Features (X)** = Input columns — jo model ko dete hain
- **Target (y)** = Output column — jo model predict karta hai

---

## Concept 2 — Train/Test Split

```
Poora Dataset (100%)
        |
        ├── Training Set (80%) ← Model yahan se seekhta hai
        └── Testing Set  (20%) ← Model yahan test hota hai
```

**Kyun split karte hain?**
Socho exam ki tayari — tum book padhte ho (training), phir exam dete ho (testing).
Agar exam mein wohi sawal aayein jo book mein the — toh pata nahi chalega tumhe actually samajh aaya ya nahi!

Isliye model ko **nayi, andekhui data** pe test karte hain.

---

## Concept 3 — Supervised vs Unsupervised

| Type | Matlab | Example |
|------|--------|--------|
| **Supervised** | Target column hota hai — model seekhta hai | Marks predict karna |
| **Unsupervised** | Target nahi hota — model khud groups banata hai | Customers group karna |

**Is notebook mein hum Supervised Learning karenge!**

---
---
# 📈 PART 1 — LINEAR REGRESSION
---
---

## 📖 Theory — Linear Regression Kya Hai?

### Simple Definition
> Do cheezoon ke beech **straight line** draw karo — phir us line se future values predict karo.

### Real Life Example
```
Study Hours → Marks

Marks
 100 |              ●
  80 |          ●
  60 |      ●          ← Ye points hain (actual data)
  40 |  ●
  20 |
     |________________
       1  2  3  4  5   Study Hours

Ab ek LINE draw karo in points ke beech se:

Marks
 100 |           ___--
  80 |       __--
  60 |   __--             ← Ye hai Linear Regression Line!
  40 |--
     |________________
       1  2  3  4  5

Ab agar koi pooche: '6 hours padhne wale ko kitne marks milenge?'
Line ko 6 tak extend karo → answer milega!
```

### Formula
```
y = mx + c

y = predict ki hui value (marks)
x = input value (study hours)
m = slope (line kitni steep hai)
c = intercept (line kahan se shuru hoti hai)

ML model khud m aur c find karta hai!
```

### Kab Use Karein?
Jab predict karna ho:
- House price (bedrooms, area se)
- Salary (experience se)
- Sales (advertising spend se)
- Temperature (season se)

**Rule: Target column numbers hone chahiye (continuous values)**

## 📊 Step 1 — Dataset Banao

In [ ]:
np.random.seed(42)
n = 300

# Student dataset
lr_df = pd.DataFrame({
    'study_hours': np.random.uniform(1, 9, n).round(1),
    'attendance':  np.random.uniform(60, 100, n).round(1),
    'sleep_hours': np.random.uniform(4, 9, n).round(1),
    'tv_hours':    np.random.uniform(0, 5, n).round(1),
    'prev_score':  np.random.normal(65, 12, n).clip(30, 100).round(1),
})

# Marks — logically based on features
lr_df['marks'] = (
    lr_df['study_hours'] * 8   +
    lr_df['attendance']  * 0.3 +
    lr_df['sleep_hours'] * 2   +
    lr_df['tv_hours']    * (-5)+
    lr_df['prev_score']  * 0.4 +
    np.random.normal(0, 6, n)
).clip(30, 100).round(1)

print('Dataset shape:', lr_df.shape)
print()
print('Pehli 5 rows:')
print(lr_df.head())
print()
print('Statistics:')
print(lr_df.describe().round(2))

## 🔍 Step 2 — Quick EDA (Marks ke saath relationships)

In [ ]:
# Correlation with marks
print('=== Marks ke saath Correlation ===')
corr = lr_df.corr()['marks'].drop('marks').sort_values(ascending=False)
print(corr.round(3))
print()
print('Sabse zyada affect karne wala:', corr.abs().idxmax())
print('Sabse kam affect karne wala: ', corr.abs().idxmin())

In [ ]:
# Scatter plots
features = ['study_hours', 'attendance', 'tv_hours', 'prev_score']
colors   = ['steelblue', 'green', 'tomato', 'purple']

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for i, (feat, color) in enumerate(zip(features, colors)):
    axes[i].scatter(lr_df[feat], lr_df['marks'],
                    alpha=0.4, color=color, s=20)
    r = lr_df[feat].corr(lr_df['marks'])
    axes[i].set_xlabel(feat)
    axes[i].set_ylabel('Marks')
    axes[i].set_title(f'{feat}\nr = {r:.2f}', fontweight='bold')

plt.suptitle('Features vs Marks — Relationships',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

## 🔧 Step 3 — ML Pipeline

```
1. Features (X) aur Target (y) alag karo
        ↓
2. Train/Test Split karo
        ↓
3. Model banao
        ↓
4. Model ko TRAIN karo (fit)
        ↓
5. PREDICT karo
        ↓
6. EVALUATE karo (kitna sahi predict kiya?)
```

In [ ]:
# ── STEP 1: X aur y alag karo ────────────────────────────
# X = Features — model in columns se seekhega
# y = Target  — model ye predict karega

X = lr_df[['study_hours', 'attendance', 'sleep_hours', 'tv_hours', 'prev_score']]
y = lr_df['marks']

print('X shape (features):', X.shape)  # (300, 5)
print('y shape (target):  ', y.shape)  # (300,)
print()
print('Features (X) sample:')
print(X.head(3))
print()
print('Target (y) sample:')
print(y.head(3))

In [ ]:
# ── STEP 2: Train/Test Split ─────────────────────────────
# test_size=0.2  → 20% test, 80% train
# random_state=42 → same split hamesha (reproducible)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print('=== Train/Test Split ===')
print(f'Total data    : {len(X)} rows')
print(f'Training set  : {len(X_train)} rows (80%)')
print(f'Testing set   : {len(X_test)} rows (20%)')
print()
print('Training pe model seekhega')
print('Testing pe model ka imtehaan hoga!')

In [ ]:
# ── STEP 3 & 4: Model banao aur Train karo ───────────────
# LinearRegression() — model object banao
# .fit()            — model ko training data pe train karo

lr_model = LinearRegression()
lr_model.fit(X_train, y_train)

print('✅ Model train ho gaya!')
print()

# Model ne kya seekha?
print('=== Model Ne Kya Seekha? ===')
print('(Har feature ka weight — kitna important hai)')
print()
for feature, coef in zip(X.columns, lr_model.coef_):
    direction = '↑ Marks badhte hain' if coef > 0 else '↓ Marks ghatte hain'
    print(f'  {feature:15s}: {coef:+.2f}  → {direction}')
print(f'  Intercept      : {lr_model.intercept_:.2f}')

In [ ]:
# ── STEP 5: Predict karo ─────────────────────────────────
# .predict() — test data pe predictions nikalo

y_pred = lr_model.predict(X_test)

print('=== Actual vs Predicted (pehle 10) ===')
comparison = pd.DataFrame({
    'Actual Marks':    y_test.values[:10].round(1),
    'Predicted Marks': y_pred[:10].round(1),
    'Difference':      (y_test.values[:10] - y_pred[:10]).round(1)
})
print(comparison.to_string(index=False))

In [ ]:
# ── STEP 6: Evaluate karo ────────────────────────────────
# Teen metrics hain Linear Regression ke liye:

mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('=== Model Performance ===')
print()
print(f'MAE  (Mean Absolute Error) : {mae:.2f}')
print(f'  → Average mein {mae:.1f} marks ka error hai')
print()
print(f'RMSE (Root Mean Sq Error)  : {rmse:.2f}')
print(f'  → Bade errors pe zyada penalty')
print()
print(f'R²   (R-squared Score)     : {r2:.3f}')
print(f'  → Model ne {r2*100:.1f}% variation explain ki')
print()
print('R² Score ki reading:')
print('  1.0 = Perfect model')
print('  0.8+ = Bohot acha')
print('  0.6+ = Theek hai')
print('  0.4- = Improve karna chahiye')

## 📊 Step 4 — Visualization

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── Plot 1: Actual vs Predicted ──────────────────────────
axes[0].scatter(y_test, y_pred, alpha=0.5, color='steelblue', s=30)
# Perfect prediction line
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
axes[0].plot([min_val, max_val], [min_val, max_val],
             'r--', linewidth=2, label='Perfect Prediction')
axes[0].set_xlabel('Actual Marks')
axes[0].set_ylabel('Predicted Marks')
axes[0].set_title(f'Actual vs Predicted\nR² = {r2:.3f}', fontweight='bold')
axes[0].legend()
# Note: Agar sab points red line pe hoon = perfect model!

# ── Plot 2: Residuals (errors) ───────────────────────────
residuals = y_test - y_pred
axes[1].scatter(y_pred, residuals, alpha=0.5, color='purple', s=30)
axes[1].axhline(0, color='red', linestyle='--', linewidth=2)
axes[1].set_xlabel('Predicted Marks')
axes[1].set_ylabel('Residual (Actual - Predicted)')
axes[1].set_title('Residual Plot\n(0 ke paas = better)', fontweight='bold')
# Note: Points 0 ke aas paas hone chahiye — koi pattern nahi

# ── Plot 3: Feature Importance ───────────────────────────
feat_importance = pd.Series(
    np.abs(lr_model.coef_),
    index=X.columns
).sort_values(ascending=True)

colors_bar = ['tomato' if c < 0 else 'steelblue'
              for c in lr_model.coef_[
                  [list(X.columns).index(f) for f in feat_importance.index]
              ]]

axes[2].barh(feat_importance.index, feat_importance.values,
             color=colors_bar, edgecolor='white')
axes[2].set_xlabel('|Coefficient| — Importance')
axes[2].set_title('Feature Importance\n(kitna impact hai marks pe)', fontweight='bold')

plt.suptitle('Linear Regression — Results', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Naye Student Ka Score Predict Karo! ──────────────────
print('=== Naye Students Ka Score Predict Karo ===')
print()

new_students = pd.DataFrame({
    'study_hours': [7,   2,   5],
    'attendance':  [95,  65,  80],
    'sleep_hours': [8,   5,   7],
    'tv_hours':    [1,   4,   2],
    'prev_score':  [75,  50,  65],
})

predictions = lr_model.predict(new_students).round(1)

new_students['Predicted_Marks'] = predictions
new_students['Grade'] = new_students['Predicted_Marks'].apply(
    lambda x: 'A' if x>=80 else ('B' if x>=65 else ('C' if x>=50 else 'F'))
)
print(new_students.to_string(index=False))

## ✅ Linear Regression — Key Takeaways

| Cheez | Matlab |
|-------|--------|
| **Kab use karein** | Target = continuous number (marks, price, salary) |
| **Kaise kaam karta hai** | Best fit line draw karta hai data mein |
| **Training** | `model.fit(X_train, y_train)` |
| **Prediction** | `model.predict(X_test)` |
| **R² Score** | 1 ke paas = better. 0.8+ = acha model |
| **MAE** | Average error — marks mein |
| **Coefficients** | Har feature ka importance |

```
Simple Formula:
marks = (8 × study_hours) + (0.3 × attendance) + ... + intercept
Model ne ye weights (8, 0.3...) khud seekhe!
```

---
---
# 🎯 PART 2 — LOGISTIC REGRESSION
---
---

## 📖 Theory — Logistic Regression Kya Hai?

### Linear vs Logistic

```
LINEAR REGRESSION:
  Output = koi bhi number (45.6, 78.2, 91.0)
  Question: 'Kitne marks milenge?'

LOGISTIC REGRESSION:
  Output = 0 ya 1 (Pass ya Fail, Yes ya No)
  Question: 'Pass karega ya Fail?'
```

### Naam Mein 'Regression' Kyun Hai?
Confusing lagta hai! Logistic Regression **Classification** karta hai — lekin andar se math same hai, bas ek extra step hai jo output ko 0-1 mein convert karta hai.

### Sigmoid Function — Dil Ki Baat
```
Logistic Regression ek probability deta hai:

  Output: 0.85 → 85% chance of Pass → PASS ✅
  Output: 0.30 → 30% chance of Pass → FAIL ❌

  Default threshold: 0.5
  0.5 se upar = Class 1 (Pass)
  0.5 se neeche = Class 0 (Fail)
```

### Kab Use Karein?
Jab predict karna ho:
- Email spam hai ya nahi?
- Customer churega ya nahi?
- Patient ko disease hai ya nahi?
- Student pass karega ya fail?

**Rule: Target column = 2 categories (Binary Classification)**

## 📊 Step 1 — Dataset Banao

In [ ]:
np.random.seed(42)
n = 400

log_df = pd.DataFrame({
    'study_hours': np.random.uniform(1, 9, n).round(1),
    'attendance':  np.random.uniform(55, 100, n).round(1),
    'sleep_hours': np.random.uniform(4, 9, n).round(1),
    'tv_hours':    np.random.uniform(0, 6, n).round(1),
    'prev_score':  np.random.normal(60, 15, n).clip(25, 100).round(1),
})

# Marks calculate karo
log_df['marks'] = (
    log_df['study_hours'] * 8   +
    log_df['attendance']  * 0.3 +
    log_df['sleep_hours'] * 2   +
    log_df['tv_hours']    * (-5)+
    log_df['prev_score']  * 0.4 +
    np.random.normal(0, 8, n)
).clip(25, 100).round(1)

# TARGET: Pass (1) ya Fail (0)
# 50+ marks = Pass, below 50 = Fail
log_df['result'] = (log_df['marks'] >= 50).astype(int)

print('Dataset shape:', log_df.shape)
print()
print('Result distribution:')
print(log_df['result'].value_counts())
print(f'  0 = Fail: {(log_df["result"]==0).sum()} students')
print(f'  1 = Pass: {(log_df["result"]==1).sum()} students')
print()
print('Sample data:')
log_df[['study_hours', 'attendance', 'tv_hours', 'marks', 'result']].head(8)

## 🔧 Step 2 — ML Pipeline

In [ ]:
# ── X aur y ──────────────────────────────────────────────
# Note: 'marks' bhi include nahi kiya — woh target se directly related hai
# Real scenario mein marks pata nahi hote — isliye features se predict karte hain

X_log = log_df[['study_hours', 'attendance', 'sleep_hours', 'tv_hours', 'prev_score']]
y_log = log_df['result']  # 0 ya 1

# ── Train/Test Split ─────────────────────────────────────
X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(
    X_log, y_log,
    test_size=0.2,
    random_state=42,
    stratify=y_log  # Pass/Fail ka ratio same rakho dono mein
)

print('Training set:', len(X_train_l), 'rows')
print('Testing set: ', len(X_test_l),  'rows')
print()
print('stratify=y_log kyun?')
print('  Taake train aur test mein Pass/Fail ratio same rahe!')
print('  Train Pass rate:', f"{y_train_l.mean()*100:.1f}%")
print('  Test  Pass rate:', f"{y_test_l.mean()*100:.1f}%")

In [ ]:
# ── Model Train karo ─────────────────────────────────────
log_model = LogisticRegression(random_state=42, max_iter=1000)
log_model.fit(X_train_l, y_train_l)

print('✅ Logistic Regression model train ho gaya!')
print()

# Coefficients
print('=== Model Ne Kya Seekha? ===')
for feat, coef in zip(X_log.columns, log_model.coef_[0]):
    direction = '↑ Pass chance badhta hai' if coef > 0 else '↓ Pass chance ghatta hai'
    print(f'  {feat:15s}: {coef:+.3f}  → {direction}')

In [ ]:
# ── Predict karo ─────────────────────────────────────────
y_pred_l      = log_model.predict(X_test_l)         # 0 ya 1
y_pred_proba  = log_model.predict_proba(X_test_l)   # probability

print('=== Predictions (pehle 10) ===')
results = pd.DataFrame({
    'Actual':       y_test_l.values[:10],
    'Predicted':    y_pred_l[:10],
    'Fail_Prob %':  (y_pred_proba[:10, 0] * 100).round(1),
    'Pass_Prob %':  (y_pred_proba[:10, 1] * 100).round(1),
    'Correct?':     ['✅' if a==p else '❌'
                     for a,p in zip(y_test_l.values[:10], y_pred_l[:10])]
})
print(results.to_string(index=False))

In [ ]:
# ── Evaluate karo ────────────────────────────────────────
accuracy = accuracy_score(y_test_l, y_pred_l)

print('=== Model Performance ===')
print(f'Accuracy: {accuracy*100:.1f}%')
print(f'Matlab: {accuracy*100:.1f}% students ko sahi classify kiya!')
print()
print('Classification Report:')
print(classification_report(y_test_l, y_pred_l,
                            target_names=['Fail', 'Pass']))

In [ ]:
# ── Confusion Matrix — Sabse Important Chart! ────────────
cm = confusion_matrix(y_test_l, y_pred_l)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Confusion Matrix heatmap
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    ax=axes[0],
    xticklabels=['Predicted Fail', 'Predicted Pass'],
    yticklabels=['Actual Fail', 'Actual Pass']
)
axes[0].set_title('Confusion Matrix', fontweight='bold', fontsize=12)

# Confusion matrix explanation
axes[1].axis('off')
explanation = (
    f'Confusion Matrix Samjho:\n\n'
    f'✅ True Negative  (TN): {cm[0,0]}\n'
    f'   Actual Fail, Predicted Fail — SAHI!\n\n'
    f'❌ False Positive (FP): {cm[0,1]}\n'
    f'   Actual Fail, Predicted Pass — GALAT!\n\n'
    f'❌ False Negative (FN): {cm[1,0]}\n'
    f'   Actual Pass, Predicted Fail — GALAT!\n\n'
    f'✅ True Positive  (TP): {cm[1,1]}\n'
    f'   Actual Pass, Predicted Pass — SAHI!\n\n'
    f'Overall Accuracy: {accuracy*100:.1f}%'
)
axes[1].text(0.05, 0.95, explanation,
             transform=axes[1].transAxes,
             fontsize=11, verticalalignment='top',
             fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightblue', alpha=0.3))

plt.suptitle('Logistic Regression — Confusion Matrix',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Naye Students Ko Predict Karo! ───────────────────────
print('=== Naye Students — Pass Karega Ya Fail? ===')
print()

new_s = pd.DataFrame({
    'study_hours': [7,   1,   4,   8],
    'attendance':  [90,  55,  72,  98],
    'sleep_hours': [8,   4,   6,   7],
    'tv_hours':    [1,   5,   3,   0],
    'prev_score':  [70,  40,  58,  80],
})

new_pred       = log_model.predict(new_s)
new_pred_proba = log_model.predict_proba(new_s)

new_s['Pass_Probability'] = (new_pred_proba[:, 1] * 100).round(1)
new_s['Result']           = ['PASS ✅' if p == 1 else 'FAIL ❌' for p in new_pred]
print(new_s.to_string(index=False))

## ✅ Logistic Regression — Key Takeaways

| Cheez | Matlab |
|-------|--------|
| **Kab use karein** | Target = 2 categories (Pass/Fail, Yes/No, 0/1) |
| **Output** | Probability (0 to 1) — phir threshold se class |
| **Accuracy** | Kitne % sahi classify kiye |
| **Confusion Matrix** | Sahi/Galat predictions ka breakdown |
| **predict_proba()** | Exact probability deta hai |
| **stratify** | Train/test mein class ratio same rakhta hai |

```
Linear Regression  →  Number predict karta hai
Logistic Regression → Category predict karta hai (binary)
```

---
---
# 🌳 PART 3 — DECISION TREE
---
---

## 📖 Theory — Decision Tree Kya Hai?

### Simple Definition
> Ek sawal poochho — phir jawab ke hisaab se agla sawal — aur akhir mein answer!

### Real Life Example — Doctor Ka Decision
```
                    Fever hai?
                   /          \
                 Haan          Nahi
                 /                \
         Khansi bhi?           Theek ho!
         /        \
       Haan        Nahi
       /               \
  COVID Test       Zyada paani
  Karo             piyo
```

**Decision Tree exactly aisa hi kaam karta hai — bas data se khud seekhta hai ke kaunsa sawal poochhe!**

### Important Terms

```
ROOT NODE    = Pehla sawal (sabse important feature)
BRANCH       = Sawaal ka jawab (haan/nahi ya value)
LEAF NODE    = Final answer (class ya value)
DEPTH        = Kitne levels hain tree mein

          [Study > 5?]          ← ROOT NODE
          /          \
      [Haan]        [Nahi]
        /                \
  [Attend > 80?]    [FAIL ❌]   ← LEAF NODE
    /        \
[PASS ✅]  [FAIL ❌]            ← LEAF NODES
```

### Linear vs Logistic vs Decision Tree

| | Linear Reg | Logistic Reg | Decision Tree |
|--|-----------|-------------|---------------|
| **Output** | Number | Binary (0/1) | Koi bhi class |
| **How** | Line draw karta hai | Probability | Rules banata hai |
| **Samajhna** | Medium | Medium | **Bohot aasan!** |
| **Multiple classes** | Nahi | Mushkil | **Haan!** |

## 📊 Step 1 — Dataset Banao (3 Classes!)

In [ ]:
np.random.seed(42)
n = 500

dt_df = pd.DataFrame({
    'study_hours': np.random.uniform(1, 9, n).round(1),
    'attendance':  np.random.uniform(55, 100, n).round(1),
    'sleep_hours': np.random.uniform(4, 9, n).round(1),
    'tv_hours':    np.random.uniform(0, 6, n).round(1),
    'prev_score':  np.random.normal(62, 14, n).clip(25, 100).round(1),
})

dt_df['marks'] = (
    dt_df['study_hours'] * 8   +
    dt_df['attendance']  * 0.3 +
    dt_df['sleep_hours'] * 2   +
    dt_df['tv_hours']    * (-5)+
    dt_df['prev_score']  * 0.4 +
    np.random.normal(0, 7, n)
).clip(25, 100).round(1)

# TARGET: 3 classes — A, B, C (not just pass/fail)
def grade(m):
    if m >= 75:  return 'A — Excellent'
    elif m >= 55: return 'B — Good'
    else:         return 'C — Needs Work'

dt_df['grade'] = dt_df['marks'].apply(grade)

print('Grade distribution:')
print(dt_df['grade'].value_counts())
print()
dt_df[['study_hours', 'attendance', 'tv_hours', 'marks', 'grade']].head(8)

## 🔧 Step 2 — ML Pipeline

In [ ]:
# ── X aur y ──────────────────────────────────────────────
X_dt = dt_df[['study_hours', 'attendance', 'sleep_hours', 'tv_hours', 'prev_score']]
y_dt = dt_df['grade']

# ── Train/Test Split ─────────────────────────────────────
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_dt, y_dt,
    test_size=0.2,
    random_state=42,
    stratify=y_dt
)

print('Training:', len(X_train_d), 'rows')
print('Testing: ', len(X_test_d),  'rows')

In [ ]:
# ── Model Train karo ─────────────────────────────────────
# max_depth=4 → tree ki maximum depth 4 levels
# Zyada depth = overfitting ka risk!

dt_model = DecisionTreeClassifier(
    max_depth=4,
    random_state=42,
    min_samples_split=10  # ek node split karne ke liye minimum 10 samples chahiye
)
dt_model.fit(X_train_d, y_train_d)

print('✅ Decision Tree train ho gaya!')
print(f'Tree depth: {dt_model.get_depth()}')
print(f'Leaf nodes: {dt_model.get_n_leaves()}')

In [ ]:
# ── Predict aur Evaluate ─────────────────────────────────
y_pred_d  = dt_model.predict(X_test_d)
accuracy_d = accuracy_score(y_test_d, y_pred_d)

print(f'Accuracy: {accuracy_d*100:.1f}%')
print()
print('Classification Report:')
print(classification_report(y_test_d, y_pred_d))

## 📊 Step 3 — Decision Tree Visualization
### Ye Decision Tree Ka Sabse Bada Fayda Hai — Dekh Sakte Ho!

In [ ]:
# ── Tree Plot ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(20, 10))

plot_tree(
    dt_model,
    feature_names = X_dt.columns.tolist(),
    class_names   = dt_model.classes_,
    filled        = True,   # color se class dikhao
    rounded       = True,   # rounded boxes
    fontsize      = 9,
    ax            = ax
)

ax.set_title(
    'Decision Tree — Student Grade Prediction\n'
    '(Har node pe ek sawal — leaf node pe final answer)',
    fontsize=14, fontweight='bold'
)
plt.tight_layout()
plt.show()

print()
print('Tree ko kaise padhein:')
print('  Upar se neeche — har node pe condition check hoti hai')
print('  True  → left branch')
print('  False → right branch')
print('  Akhri node (leaf) mein final class hoti hai')

In [ ]:
# ── Feature Importance ───────────────────────────────────
# Decision Tree khud batata hai — konsa feature sabse important tha!

importance = pd.Series(
    dt_model.feature_importances_,
    index=X_dt.columns
).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
colors = sns.color_palette('Set2', len(importance))
axes[0].barh(importance.index, importance.values,
             color=colors, edgecolor='white')
axes[0].set_xlabel('Importance Score')
axes[0].set_title('Feature Importance\n(Decision Tree)',
                  fontweight='bold')

for i, (val, name) in enumerate(zip(importance.values, importance.index)):
    axes[0].text(val + 0.002, i, f'{val:.3f}', va='center', fontsize=10)

# Confusion Matrix
cm_d = confusion_matrix(y_test_d, y_pred_d, labels=dt_model.classes_)
sns.heatmap(
    cm_d,
    annot=True, fmt='d', cmap='Greens',
    xticklabels=[c.split(' ')[0] for c in dt_model.classes_],
    yticklabels=[c.split(' ')[0] for c in dt_model.classes_],
    ax=axes[1]
)
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')
axes[1].set_title(f'Confusion Matrix\nAccuracy: {accuracy_d*100:.1f}%',
                  fontweight='bold')

plt.suptitle('Decision Tree — Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── Overfitting Check — Best Depth Dhundho ───────────────
# Overfitting: Model training pe 100% accurate, testing pe fail
# Underfitting: Model dono pe kharab

train_acc = []
test_acc  = []
depths    = range(1, 15)

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=42)
    m.fit(X_train_d, y_train_d)
    train_acc.append(accuracy_score(y_train_d, m.predict(X_train_d)))
    test_acc.append( accuracy_score(y_test_d,  m.predict(X_test_d)))

plt.figure(figsize=(10, 5))
plt.plot(depths, [a*100 for a in train_acc],
         'b-o', label='Training Accuracy', linewidth=2)
plt.plot(depths, [a*100 for a in test_acc],
         'r-o', label='Testing Accuracy', linewidth=2)
plt.axvline(x=4, color='green', linestyle='--',
            linewidth=2, label='Our Choice (depth=4)')
plt.xlabel('Tree Depth')
plt.ylabel('Accuracy %')
plt.title('Overfitting Check — Training vs Testing Accuracy',
          fontweight='bold')
plt.legend()
plt.xticks(depths)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()
print('Chart se kya seekha?')
print('  Depth badhne se Training accuracy badhti hai')
print('  Lekin Testing accuracy ek point ke baad girne lagti hai')
print('  Yahi OVERFITTING hai!')
print('  Best depth wahan hai jahan test accuracy maximum ho')

In [ ]:
# ── Naye Students Ka Grade Predict Karo! ─────────────────
print('=== Naye Students Ka Grade ===')
print()

new_dt = pd.DataFrame({
    'study_hours': [8,   1,   5,   3,   7],
    'attendance':  [95,  55,  78,  65,  88],
    'sleep_hours': [8,   4,   6,   5,   7],
    'tv_hours':    [0,   5,   2,   4,   1],
    'prev_score':  [80,  35,  60,  45,  72],
})

new_grade = dt_model.predict(new_dt)
new_proba = dt_model.predict_proba(new_dt)

new_dt['Predicted_Grade'] = new_grade
for i, cls in enumerate(dt_model.classes_):
    short = cls.split(' ')[0]
    new_dt[f'{short}_%'] = (new_proba[:, i] * 100).round(0).astype(int)

print(new_dt.to_string(index=False))

## ✅ Decision Tree — Key Takeaways

| Cheez | Matlab |
|-------|--------|
| **Kab use karein** | Kisi bhi classification task pe |
| **Fayda** | Samajhna aasan — tree visualize kar sakte ho |
| **max_depth** | Zyada = overfitting, kam = underfitting |
| **Feature importance** | Khud batata hai kaunsa feature important hai |
| **Multiple classes** | 2 se zyada categories bhi handle karta hai |
| **Overfitting** | Training pe acha, testing pe kharab |

```
Overfitting se bachne ke liye:
  max_depth     → tree ko limit karo
  min_samples   → chote nodes mat banao
  Random Forest → bahut saare trees ka average (next topic!)
```

---
# 🏆 FINAL — Teen Algorithms Ka Comparison

In [ ]:
print('=' * 60)
print('        TEEN ALGORITHMS — SUMMARY')
print('=' * 60)
print()
print(f"{'Algorithm':<22} {'Type':<20} {'Metric':<10} {'Score'}")
print('-' * 60)
print(f"{'Linear Regression':<22} {'Number Predict':<20} {'R²':<10} {r2:.3f}")
print(f"{'Logistic Regression':<22} {'Binary Classify':<20} {'Accuracy':<10} {accuracy:.3f}")
print(f"{'Decision Tree':<22} {'Multi Classify':<20} {'Accuracy':<10} {accuracy_d:.3f}")
print('=' * 60)
print()
print('Kab kaunsa use karein?')
print()
print('  Linear Regression  → Price, Salary, Temperature predict karo')
print('  Logistic Regression→ Pass/Fail, Spam/Not, Yes/No predict karo')
print('  Decision Tree      → A/B/C grade, Multiple categories')

---
# 🗺️ Aage Kya Seekhna Hai?

```
✅ Linear Regression     → Numbers predict karna
✅ Logistic Regression   → Binary classification
✅ Decision Tree         → Multi-class classification

⏭️  Random Forest        → Decision Tree ka powerful version
                           (Bahut saare trees → better accuracy)

⏭️  K-Nearest Neighbors  → Neighbors dekh ke decide karna

⏭️  Feature Scaling      → StandardScaler, MinMaxScaler
                           (KNN aur Logistic ke liye zaroori)

⏭️  Cross Validation     → Model ko properly test karna

⏭️  Hyperparameter Tuning→ Model ko best settings dena

⏭️  K-Means Clustering   → Unsupervised — groups banana
```

---

## ML Pipeline — Hamesha Ye Order Follow Karo

```python
# 1. Data Load
df = pd.read_csv('data.csv')

# 2. EDA
df.info(), df.describe(), df.isnull().sum()

# 3. Cleaning
# missing values, outliers, wrong types

# 4. Feature Engineering
# naye columns banana

# 5. X aur y alag karo
X = df.drop('target', axis=1)
y = df['target']

# 6. Train/Test Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

# 7. Model banao aur train karo
model = LinearRegression()  # ya LogisticRegression() ya DecisionTree()
model.fit(X_train, y_train)

# 8. Predict karo
y_pred = model.predict(X_test)

# 9. Evaluate karo
accuracy_score(y_test, y_pred)  # classification ke liye
r2_score(y_test, y_pred)        # regression ke liye
```